# PINNS 历史影像地图坐标修复

在 Kaggle 私有 CPU Notebook 中运行。将同包的 `pinns_map_axis_inputs.bin` 添加为私有数据集，Internet 可关闭。依次运行三格，最后下载 `pinns_map_axis_comparison_results.zip`。计算原来和修复后的轨迹误差后，输出中含私有逐窗资料。


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, subprocess, sys, tempfile, zipfile, zlib
folder = Path('/kaggle/working/pinns_map_axis_comparison')
folder.mkdir(parents=True, exist_ok=True)
assets = {}
assets['compare_axes.py'] = ('f415ffd3696ed996e414304cbd72a276f8f6c32283d219c1fd8cd4cdb841af91', 'eNqdO2uTm0iS3/tXcP6wgE3LQm/Jw0T02e1Zx43tjrZ3NnZ0CqKAosUIAQtI/bqe376ZWUVRSOq+mekIW4KqyszKd2aVXr16dcWSkkeOUeW7MuTnaR5ueGTwPUt3rE7yzMhjo15zIy+TmyRjqcGyyAjzsuRhDRNDtqvg5dWnL1++GcmW3XBjy4qq9+rVq7NkW+RlbbDypmBlxdVzVZ/FZb4FKGkKUABJZcix9/kuq3nZTF2zap0mQfOY5M2336o8E0AKVuOUBsAVPDaTipTVcV5um+eab4s4SRUh9X3BKwHlIaGRBsqvn678D5cff774fvnBMX5Nio+4rFmX7bbFPWzDyIqzs7Pry58+fft+ffH909cvhmeY09FkOh/Gk/lgNuGTOIj52HVH7njGuDsau2wynQTTyZRNR9OZG48ncTRlkQtf2ajvjvpj8+zj9ddfL7/4V/9CcCyMwjDox7NpOJn3h+MpH49g5WA2Hk0n8cxl7twd9kfjKOzP+n2ANeLxOIrHs0kQBmzGzLOv159++vTl4mf/+uu375fXCHQynU5jdzYY8eE4HvbjwZjP5v0gGI6jIJjPZpPRrO9G4zl3A9jFdDqeDsN4EM+AkPloPo01oJ8vrvyL6+uLf31DwIPAnU2HcX88ieKADUfBdB5H8+E8YqMwnjA2m4fD8SDqR64bT2eAqT+Z9aeDKHBHwZjP+13ABHI0HY2HM3ce9MejQchjFgV9dzCYzMbRmI3Gs+lkPJ8AE8KBG/XZbDyOYBfwKpxxPov5VGfA5cfL68sv7y8J8Gw6nw2jGR8MWX82HvIY5BIEfDKbj+fhNBrySR+IGrrzaDSbxOFkMowG7mgwAk6N+rMg6gC++vlCiCseTV2Aw2fDAR9Ng+mIc96PRoMhsGYMEnGn0+F8OJgMRpNZGPcn7mTCWcB4NO+H4ZSbZ58vv//96wek0DK3OVqH6QDcPUtSFiRpUt/jM5maf8PzLa9LepOH4a5gWXjfvrXPfrm8/gZ6KaCl/IbBcAkGyEtcImzeB7g8hcmA+frTezGXRdzf4pw4wglV7YNHSKL2Hb9jYe0PK3hjgxVEPDZK/u8dOBMr3zjwnYGJ2oszA/6S2Mjy2sg34hH/SpZU3PgFvAy/LMu8tOQCCapaM6tkt3J9yetdmTXeoAeDg/GExntrfhclN7yqrWbpbzjNiljNuqsRJLqNXgTmW9EE9Hpl7W/4feV9L3ccnjm4KlbnZeVZpoN7XZi23eNZmEfcshsc4DMeeOZvwdlYOCIxgdP7B+wK3WVwXytnWuyCNAnRt6R8y7Oa/KpjJFmY7qIkuzGSutL8a5nUa5AfrEBXG+R5XdUlK8iliv0INuOGCLnheYbyGUDy1waU9OJ5acSAhpdFmWQ1iAuhxLtMul7PeDQblfFjzoBbvCKdK0t27wv2ktiT2l8DnQ+wip5zCAJMjHEUIr0N06TwE/TigN18ImQhrKhZVgtkP11+/YwzL36++vsFfkHz+fQenO03fPp2eflBrtvm0Q68socxo0dBROyYBjP4hgCXmQGUGBkwVC7oBXl0j1pnKX1LqoRICLkFrEdwHyUDPvDYJk5nvYxtOUJRvFHLAf4xhIuqSm4ysTjlmZX1aoh1vK5IIi6911c1w8v+SgD4Avga3O1YL4mQCMUze9URO2HCrRMWfFLk2sYbetEuBXZ+JF01QG2TLKFpUgN4tidxZIW5gFAGU6WBwaP8Bu/QZOAFfgiZ8DseAoZtASHRwl18Jp5byHOPCHMosvrAG1CQyluukAxpMlIluV8kWVb5In1QHqtXkCtDFCYsAgpt3YQpYPe+kSEh76qCAV9fv37cLHDucrMiTdh0ZGj8X8vKp8aAg12SRkrZLeFnAWEKlhOk4AjIvzrGepem8gGsvdqIN/RV2jwiBEeyRZyWJZe1C2AXlgDSLrQ1NyiFmhW9pCIBAU/tHktTSypGQQ9s+ft2hfLuy9fgTq2t8Tfj94ZkWJTdW8jpzwnoJXiVhqmovMLTg3KwjZR+AfsGt4bGBTggFdttMx+4FG4s64ANRzsXEAJWnVzdMLPBIKcjs9FcoySs5RyvmapHNw/hNpibPXhHWLRZtu0ohqq/43D4HAzcl213FK0h3RFUS6UpKFX2QcNBtS0QOtgHJrv4f5bE4CXhK+pdhfEPss/7Ni5ccxYZeZbeG+ukAo+chOCfG/2DfUQQF5I44WXVpOIGEM0wSpHAdQ4ZAUfXi86OA5CkrkHcR+FB0bRkq6VZlHmdw+7NleeZ0uoqMCGf3DXEF9C97MYvecxL3Lm/d9ESf6aUwWAhwKzIexgKkv0MuuAvoRv4nWykI1BkQBd+wPc+WPUeuQb+CbCgr3qvypI/RTDyRyCvTDKyLq52COkQ6cUzi8kln0g3u/ppXmT3RhXyjEtRg8OUHIHMArjkQFkF3reUiQLYLzFFj+JGuGbwEMlNYQoDtpWCbllQy1iolkQa8ttc9eqchmyhsEIjQNfyzE+i08N1X3tv20chCDHanodfK16LR3ieDmZDjU9y1kEJAqL6Z5JF+S14ZkgoVEpCiQpPKfxZgnSY2hIKD0AWJq4F2IGp+VEw7RrEgXbAqopDUieSFyCYpYobBHzVGKd8fAG3jpowS8ROkzk3jhKn+hCBfPQl8LAFP1mIh79MJD0Hqw6VSBbK+UC87wyiqzMiKMUhiCk4Ijdt4rPZTSlk1W0JKK3UUZsfTdJNczGezCaOiTndwnX7fWAIpoULKAefVAaEmkA0oiqMpp33RAe9P7AGla4SeopWMNUocypTuprXIvA8/P6cHYKO/CPbZPltRjyTUPYaG/YcsnMspoBFIgxpg0lW7GoobUT51WXWHuuPgnuehbruDERAFlO7Q+5IxXAV3MU8GeH1XD1O9vycrF/CkjS/rDR7R85eLhzjfLDQtBn3rWIJpsanWdUDqraVpalpcucRiz36OEpVhCCXyR04XQGl0al2V98EVpJjlMTg5JukU0CK1co2Arj99QMKA8q6dpTd+iIRIM5UOL4Gc1ATOCvTBDbl41uIK8kNOEgJSuHjdwVFBa/j3HAHP27ZHVSeUAAkWL7GkABCTgMsf+CgflZy16t2W9xRhKmnB7VYav9Bc26Qai710GkQCS08QcKiYxy6x0U6f/S6RoU1MHpdGSLFZs63iZj9gzfpgzS+5FnIsjyjnEMxHGrdWhOKjm1NfrAyYCE2/hTgH3Dg3J23EoaqPVyDe6EVIrO5PwAK22p4IVLUxVGy1obkO9rRD97wtXUkrGUDZtXDDdrnLqraL5A65SJsGiwGj6EnWOGuzuNYEiRTqJR7j0/vVO5FDyL9qhN6pMkyZ9+CU6CMrndUIbcEaq4f/Pa+NcGmnmy6MO3OhToLhcRZqz+oVeutI9cu9SCzgrmooRZpqKZSTTniPbeJBpjYA82HTez/qJYnW6dLjoTwDD2N6nvNoiOT6CxU65ToBLOchnzx6J2u5lQVI7hAEgV611uNP0ruEpJgk974kBzqhiFtHVtp6Y3WDKXiv50WrP7L66yidDVNQQGwi65S2YpaDRk121RP6A+FgpZHALwJZJ22oRYcQAvWOeU6st34YpaSpmGaV9xSbAetFxBaTDJkmm/UAKQRXt9h+OHyc3fQqbAowjgi4XAoojhqA0q8TssxWYKhilW7AtvwlqhK1SrpV5gTeNaBvixx6uqkSQqiEuc3j0EIdVdOQB86qY/mLSWrlbmAxJvQirBgO+ZxJSMmIbD+Sk2D9bzgGe7GV4VpO3OgZnYzIxH6ctBtrBzFfCvBqv83u4VNkzJ+q0/6HWe1k06BDfJ6rYE8ATGp15CAaQB/fw7iLUzNd7XfLdqRcrW8Bf/MZNhBM1mheWrawSHUu34JMrB4WWJIVf0XMAEP3gkRS7FVu7RuHDkKPXEasfNst+VQV3FLNru1aISdKfDJABBEkqze7Sru6ambGLffZR5SCaOSSC2vQcTSkzyaMOpvzUWc5qyWi5ewaqUxQqpVBhUDZ9np2TiCuTg20UV+gmH5SddQgbjpTYNFWbILQd0LqCb3WGHu6rVjkGOR9abce1FCavRo5mnlYxJqLsI9hETsg4NgIHPTOSmM27mjDhsCP04fU8jJMoAoXKneM7buHKLiHOgRdAgy1FKkpHEszXrZY4a1sErC1lYoKnu7IkK5Por1b0xwRZvFXjQEnT0SLFc3JD8dtnyALy08yU1sdDLQPdLAyirkaankr3Bc0pkJV6RYLH2bYLzkD2p6EgJHUUcWj9uF2KRonlsPgj2CN7bgt/OAlCNxitVPypE59J4mEF1qBiED63PAqDxLYl1mq7ZR3vV/EU9rhubqwb/l4eHS6pzedmOJtg6RwL8T6+jtiXUQ72o8hgJOmOosWToEyO12ld++7SxfKIxHLb/meEuHIl+dAoHcOQIhahUqR5LMb58qDfG5Wq+ZxclIqna51DbZIQyi5YpYtxU83jZsjY2bMt8VluboOhFJxkBzcRQQaa/gXEgnzUWnKhdO0qZ2/A4jrF/k4Mt8iEs50IUnRcIFkwaK2ScdPjFLHvksHveoybqLdohsqcBcnsvwtKvAe0f0L3FUKKgaP4FMKDgg2+iY7jRMG+GVFNMVtKdnooJoN1C7B7sasqGhcRthe8KMPfF56OrpJfr6AtJVJE+TGjA5uPepMl48VtoIBFJRQwNQW9BONoxnkTzSeyUnGNHwA0BTY2PxGB6Abmp3CTo8AN1RBMkb3LjaqGCDxjMs0m8SbMaA+uKgOt2DetwDgQgn1jn3s1rd36yWuAjSIZHn0dPKEXTQd1ud2qjtWn/OLTgvWK79fKhkkFvfP3CrKJM9nkk9YMcED219vFXiqPwbvH+ZFLWjWsly/LUjMhOpM7jSoyNmCOJRZSlIPSiTIx+PhCvroJWKh7gdqN25nqdfLwGVuip5K43nWtsCqNrTIUikC1v0YgIl7r44U8f6FVDQgBHssgivxIgc/F47N648/QD8gEsnNpvIDO743Auv61jm2w27uUn529u83EAybdqQdIEMS+olXXOmDjmoqkJQ9L3E/kOFrgNb/nVu/A+BkYRimtlc3dG5YePNnYKFm2Nq8C0VP9hCt1UHhJCC6dAwxl3ZHhUtSMHNLd8GvFRstDU+4jK8FyUamm/fX7/XOiRU1NBaCyEfNEeCNA88QotMFTNOdmxQ4jhZSfeAHlk3A1GSGP0owXxzCjCZC8LU2ISlnhCtJ4lWl0porJcVD51GX0udvrhzIkBXZ/ROqJCixHZwziHcsmxkQhq10MytYeMb820zpUen19ITdvKeJw1coVLc/79xI+6u3HqazAhfUR3s/Wj/7Ba23TZhRWFqwiLd9j6zAvTlHjdkvFU7NaKkYjclp9sjBzhIzaFMwSVWkvf+Gxn86Svhc6Byz2/9Igk3Kfc+gsPmpP0PC6PtPKEHf+gcnT/08C5c9XR23J9rqG8Ys+yvDjbR7YZ80wULOMFowzUUGQdifXYTutI8txsxSUu7/mRrweue675u9ubQ2W57tCtyfnmwqxkwbaw6YpaGoLFJyNVEx0ZJ3LwSd4PaC5XIpJpnFZatXc8rTRBCOSQcIgiIE8IG3KITK047+MWLzl+6jepg9oEzOZGeCdevlqHCk2v3fVQl37e7UcExVdBQd5TU4vYqkylcpho5PEY8pqM9TpAHHCfWtqeyxxJZtFJzTLrnCZupS7yf4ft74Ayy2z+VDRf3UHZm5qK5dNoTL5pFEMrMZkifJL/gMIVIyCHp8wQGoZ+UTOpZv0pQWy3XkkpLHt85nRz36Uk/dcDLJAfxuZMxIRrH0Fwklale0/mjJwgAsBVsH6zeUfWghmWlgRknjnnHB3BHpkMQu2dpw35z0kbQDwfFkCXHoGJx7R89d3xwk0a1ckRNfnwIJy9haQYX7r39cuFgHrB6bfV77mu8kVOi87JcZ+ja9hLHHDnlHXFby6WJ+VqDg8wOXa5kg+53Xzh2xiqiPX1uTzgEW51DGTgdrjstr+0nrXWSUOPnSevcUCgvONscBkSpxc/HRAwlnpz1xvT97gUOaq5QbwVSx7Y/pfrJcuEKezxaD+SwP3TQhSqa1shxE+ql+1VFK3iQmXhJYJJQvx53cIYlSfQ8LfQtTpZn6rix0fQWdtseP7nyhdZ74agDReqoD/j5TLTY4Zvbt0/C0wTattWowYhKHFRWcd5AtXt4+GYfw5EquxSZzqmteMXZiQWqH0cLN4s7vUgXqnDQiHsRQHu+cADqSIdehir3QY6/05fy2jODBlJ3PZmLGvOQEx0TETcA2zaq09zePlDLl09xJJpOfrVdOYevXXrd7hDMSSRUPhXeQMShBUvyDsrmU5fGu+Te9Z0712uPfvaIWPiE4z5i80dXvbBD41l3fc+7c4XJsbuk8tyj42E1e0n2vhLmeWxcemZuUcccfSJ4QWQ2dR5CXqB/vOXJzRrd+OKPmpkuAakomgjIob3ZUk9ZFFJHk9zjSce29FdEv1TceU4LtBldlMdKAbO9x/bQStqUyHlCvNmDx67WaXnYzzajOlmKbBBhWtlg7xyfWdbvCr79N6vbdbLbg59uxovJcx2u260AcB5usAF1vMsTa+98cHdN11Xz9f6+8lUqugVomss8AiPOAGDS6UMBEdNEwSFqkLbsEOFM1iQi+zg7nWdJzVLnnXjTQZ7TeM0vqXoX5c0Oa8ErfCotyAwp/8b45PtRHkKK2l5CEOZCIgHbwIzebPPvc5V/Y97RtJNgfFfjfM2CIGJEESiswGyZ5+eg5w7di8FU35GhNqJfUsg7xkdLiGOmw4j7nolpDPeBH03ThnmFuOePq8CJn7X8bzjjNV071qM9OayHe4KP45oCXjZ7cgi1x3qiYyew9cQ2ESUQ2NtusOkkHuQvQji4rNrPN9quqFhtfkvW+87xR2GsvP+QYAWXl/cWAPEOQFOhWm8LreeU5zUxzoLX+oEWFf9gpewGKv4bDITiCr7+4xXJkYROlL2BrIwzlsmy+I35vxB31M9WTliu0Ae/LXckFsHMTrXW6kVvG+EM9diZdQJHucsyXuKl/sUL1eCpYpDWnJJoZ2WnipGSXRx3Ryzk9VtV8+qtql5e8MwybwOTJASlHmfbxSnHXbE9f/DR8sHkK2ym01zn9WsJ9yDdEHJcnsK68l6iSN/gUc8LKwcsrve2frApkXWP/zpUiPJWyHjlaZrUAHa6v4Rqdeesw8qmnyroF8303gMVLbem0/nZZMOqlEP14Y2Iv7L7szjK6LOTG1k0C3q3JbpVLMT1u0kvk9TBqCqBBuJRm9cxv5LFGt0G7Qm4PZJXiB5IGHnziwqMb0dW2saNFw9SNBUWp/zNaln2r2wnTnfVWjqiszOY7/uYaPg+hE/fx1ABGYg8aKa4cfYfOeR96A==')
assets['test_comparison.py'] = ('2b5e0a015dc40c63ea31a2f6ccdf5f24ddd4de5c33a91122f0617f3689c3d290', 'eNqtV99v2zYQfvdfwWVoJSWMYCdB18XQgAHtsD2sLdZhL4ZB0NLJJkqRGknZdbblb9+RlGQ7SVNgmB9k6Xj38X5+os7Ozj7ulduAEyUBVV06fYl/xIF1OXmnCS9LsJY4TVowVisuScUdJ9oQA/hQawMlt444btbgbH52djYRTauNIyhpubEwqY1uSMvdRooV6Rc/4OOgqLqm3RNuiWqjbqkbtATGP4MdDP54+9vHX96/+0jJr29///n9G7xZdUJWrAbuOgOWjma21OG5Fs5fjb4DxRrckJKGt8x2rUecTCYV1CHU1DuX3U4I/upOlbY4MgqLOUZbsdUetdMsC4pGrQvV5oarSjc5QvFOOobSdHYVNVRxfTUnIMVarCR4ZY7aa0hV9uLmm2I6J5tOymJQIC9JeqJzjToRSTR8DdNRMy91u0+zeRQvLmfL4icuLUTB7KFegGi0E1oV6F+utGm4TK24gyJVdHbTR7R5avVVlp17Nxe39J1WsAyaWlZf0u196rXnCnbPas5OcPmWC1n89c+8rvEa64GtpngDdA26oQ23n4hQ5E606dgRKbpDcaOMpnF3GqGzvqQj8sIDLWldx5vitIHSmCI6ZC/uuAl7Up+CbETDMvm+ERgYtxaMY9wYvmfwZ4dBDviLJPjBPA44s0+Wi/sAdvnqdkn70nqck9b4obi6mW+5LNJT6ex19vLea8+d4UIV9ykq/e0FEWhtdNdaD7XbgIHUr9DEXxM6CtGEJnhJcMWjJH3p+5kojuYjAoS8RZ0Y6aA6BId5Z1pJDK4oZgSH4aECFmdUmJ4q7ATOzs76lddhi3IL8ql2ucqyebkt/PLQL+dpPjs/ZGhGr7HeC79EjzvKmc5t0PDiIWq5ze2Gt5D5XIvK5w2NbIq7XU8zutJa9ks4X3QcsQD6LXmDTbICwx3IvadGbKAV4VKSnXCbDchqIMQQsOQrkHZOkGg9KTmoiO8eZBfkt5IrpR0pNz6OPKYh3FdFcH4Y9l64uA+FW14U30+n+TTot35kkPp4Y4/HZouUjQ3t52WYlcNAtIt+eUmj5fhceN5MAw/6URnVyi0NDtGQFRrcOIwElw4brKLsGfM+hKcBhkH37qaJlpbVYgsJPe/5/miYj7oRyyZsLZRwkB5CivOX5ViRNDsxe35y+yB6nniEN2a2eehkErkjyW6f36BdDJVYTJfLReM3GSWzIHm0SxhBvhJSuD3upMuya7kq9wdeOUqNt/sE+2i5A7HeOItGQmFgJbTOuwpc4Z8tuYTkQVa/4n1slNMQLhKWXOCWYx+dxjMu90QTuQ4x00VisWq4ap1JxYurLDgvvOsD6y2x7Y8NSina0eDmSYP4fupccXoaSFsaW9LSMpJa34WRNmNnR+MWRx+HDzEWkT6RyKMsWc5rYZCv4yOKuTBejJ5pgychnGwWWa8RqrPsID0pIurzCnWSyFEVSMc9Tz6BK2HNsdTHoL3oC4iLUN8Rux+TARiM0QYJ97SGQ/fiXY0o8JmXjl3bCHdE0t/9Z0iJZ0QWEv4INDI/cijz1S1s16RG7/6H3GJ/dY2/CW2CmL5RTqq62oc9k2WOrnWHo90TUyBlKbVFjun9pKERxi2+aterD7W5HGtOg0cG1sIG7mEegYV5Rad8XpNK1DUuqRJQzg5PYXF4kRy8aA1ap8kH7wK+bMfzfTwO34HRZGAOiimpoMUjPygXTqOUxLz7r4FWI1B8g2HYlz5sPPWXWuH8dWXkOzxFi5ow5gmSsaJIGB6a0UuWRGLBt2AxfAjkP5p11+BOH/yTwXcaruS8wgr2C0ipBg9f+JnBwJeEO20S6vYtFP6DIZuH07q3CoDeDouWPzbKJv8CK7lhlQ==')
assets['input_lock.json'] = ('a34a48d26cfa9bf5bef560f85e674f3918b1b9c71c686b8b792b13fe82028c60', 'eNp9kt1u3DYQhe/9FMJety45JGeG6QvkLYyZ4TBRsatdSLIRN8i7l3JsuGjdQncUcT6en+9303S6rfOT7P4wL7fH/WH7KlDw9Gk6NYUCaJUhoYcskLG2kmNV8YZOKaL6+KSCmzGwN8ea2IWhAtvpl0P94hf19V32+zgcx2f/Ivb8sMq2+/rbRZa5+7bf/7Fdl4PdMxaolHKGWsjZak2FTHMpEgoXxOAJK0iOAYXQJXbGhGS9QAN9YX+AuW33y+3Pg5Apl8TDTCgZzLs0DREAubQiuTBhqRhrNIgtCJfS0vhT0didu9M/CX47y/ObuPRMUUmdE3gmpezuoWVIoLGYSiRKNQGOTNl6wIjDwpFrDWbkb+Lb9XE1fziL+vnfIZEBBoNcQ8/VrcPogrS3HgFjGAxjRmgoBsG5JEnGlQeMbDil8B+U94zGi0XVo1gizw0rZQzRcpCGpbekJh2DctSszQQTE9dRcyFJEEs4Df0fLyu4rvOXeZHz68hkta/zk/9tbEc+YzXJPDmIG3BH8JEKt1I4hEo6mAYwRsmZIA63UUNSZK0l/qzjHbP6/rguH3BCTZJbrCF2T2O4NXLkSsCkjNZaR0ZtGVuUgEkooFFK1smkcG/yOmq5De1Vnv9/1wdwFM6UeijYukrKSrW3mmqTbB1FuFoakw0txj7S49EPB4KmMWvx+mFJh24x622sPw/VMdNEyLU2ytKsW8gjlyQ9AFiy4aLTUFNVyyqBmkR572af97Mfkp/nbR8JmpynYXCy6/Lkyz5fl0mWNv18wK8vD5jk27yNC+vqdlz4fXpLfnqNfFr9IvOyTfPl8riLnv3+dPfj7i/Ls0rc')
assets['PROTOCOL.md'] = ('746793f69286e6fbfe5114158ae1451a676b767a74781f56fd7ad181fa401405', 'eNpdV11zGskVffevmKq8OjH6BD9md51KKqnNVm0lry6MscpV2jjl9abKeUISMAMCZiQjIQkkQBICY/FhSZZgBsSP2enumSf/hZzbPRpp8wZN9/0859zL7zTeaDI9q/3wl++//1Hz+j3hbPDDFrctNv3MNkxv2mfOB2aVxEGalerM2GZG1s8U2WDMTouiX3n0iNfO+e6YNw1W67izPh3nDP84zb9ssu0C0xvudMcbDNhN37XbolxnuaLYd7z+8NfUOr+YiXKH3VywzDmbpOBEW4xqIqf/mlpj1ao76uI30SiLj9esNhTrY1jUoo/nYwsafhPdCjNPYEbFrS09Xo4t46XfLXiDNW3u8Vwkgq8IxBvsaItPo3S1vc4GdXdka3MR7c//1Vy7TF5KWWZe8Pb066SgQqcbCxGN91r3YSL/9qav67xxA7Nes+DaNh3Whv7hMTu0eIPidkct3pjwykfXLrHspejtIm4qn1VAYipUZq57qQ02alOeRpYVbIqz9FE08LzEig1yp49YIUPeT1LICBb8St8/3uN7A17IoUNwJ3q3yi/ayHsn6BMSUMlo3zz7p/awid7sSDg91Q9RHaGDFKA5gCt3WsQ7b2PKrDN/v+hOS9SSSYqZW6p4qu/uKE95jC/Zoa46iuhQAOF84Ec1ZBw2EtYIFudNd1R0nROv2RGb5+LTZmBNthJvAZ5KG2hBsX+DgdXkSjzx/vnb+M/vkm+pArtD3NJ+fvPL20Ty+Wr8RXJVc6czGXnNS2UUiPGSd7rC/uIVNlj1ikqL2Mp1VYmhzQYjwMebXvG9BhumqVLDrKqQsGei7bBMi6p/c+G1T/iRpRocVNkquKOUqrW4dCimgYl2ug7yK/lnuwCRaq3XPkPZ0Frvy42nX7LTA2buLov2NvmrddRVsbPPdBv4RcxzT3FZ9QzU4EaX7/a+Tqq4GziVWOE1gLaHwrNCyk/lYJN6aBWibNJAfioNmdIW2hqFD5Qev5NtYx+fuWm5t1VWvaWeZTr+RmcxigSoaDJDAqAsIz7AIOFra887k3jsH/n7mRCzftPxU5bfHCM4T0ed193RKbqAQ5bZEFcN3iB0KXb6TZvZpoQ5QYg+KDt3wMOJwkZQPsUqiX1VbmbmWeZakYN8SYnBc9dp8fKYmXsPQU74/JxCcg/srzMr7eldIAiYpAt3P/FOkx1tKiB5g1Z4wR3lqH5ECoNlDOqKlEQKXhm/A014EqTwIPivkxy1o9rgvVMYD3w5LUpndMZrudALhSQZH4imOQAT/QNTeQmjpWp/uVEpS+6ECIEZ7a/xlZXVpPbtD//QvFlVdDbhnLyV6t4g7U6GXvoAyWnxxMtE4kXkVSyaWH4aWViKJpcWF6Ox+djSYnT5VWwuPvd0biGyuPQyEYlFIi+j8cXkq6WXr5Ziyy8SL+KxuBbMAX3Kd4a+XmSOjeYTomXxqNiDMagRnkglyEFlCYgyFVBf+/vfftQkkOs0IqpV0S4GArFZZ5ZBoOxvQl/YRY9Vj9gU5DG8mcXyHRJe+YHlGw8xxLJFOsn0gEteS3mzrbBJSvII38auuqZ8BSCTT9Qd9AwZLEQQPa9c07ix0gob6isyEbU6S00QoIIrUbiX44cbKnBSUb3r2Z8CmQddUxOkyFOOO/1AYNazlNZ0G22T8LBZ/4BvfBTTg8gf5qhYRte7OSS+ykNFVrQTJ6C9d000pCElR1w43ODIdTLotKQYaTKIDr/hPCZVlUU386p8CmugMusX4EFRGbIDQaancvYqzNLgUcCcZXCf9ADmp9u8eUJ6dpwGRXhBRx64KZxDAECbT/4+ponPl0RuOWroBAOX9ceUfjZz56+AiQN9Yms13jtG5H71OiCUPXNnh6STpTrmrULSb0ZYbsxu15SSYZnAQ3dUprUEULunjI1zmFUNJxr0m4Tgi57KjRUvQX+kISY20kaXg+k/risaKhElLuHDxAbBeXXGi8c0scwzZBHWyLv9AEmXxHRHDoTT102M6rADqlF+NUuLiKakX3v9U3wl+Xwl+ean5Lu37zWmm1r8P/HXmHCvV1+/e09s0/743TOqU3GHpIJKuk6aFLKsW8FcADBpCbjsEHNqXZKrXot/Xlezmp5n5comq6UiCZFLs+y8GQaMWUuwqrQf8tqb7dNnOXGk63Kw0EDjsdeNeu6Y6BpojplXWQMqar7/X56w/zBLfH2TSPzy7/i/MPPDWhCYqle81ArdyEETVFbO6XvHSDLY0ay02i5oRdq5woCj0pQB6COen6rFTZEOpA5WSiwJclRBhJEYLsxHHkciERQFDIIO85oNfWI9az4yv0xh7d/SMjQtYm3zKzT/2PiaGRWeb7GtPJEDtm7Rh0/EIbjfMcKe4PmfvntGirA2JDU5P1+I0PGdqixgUcAFitJK01p9uaNEhHpSyIVLMLFCtVGOcJXAE+WMbhhZPsCQzHPDQr/UsoZnRAWHVg90mZheGxLZ7+giwZN7wisN1z5lZg5kgcoThyU7UEM/lcIJDAUKKscWqMrL2EnldIZwNca82KeVTa2FhS25r5AP1BAEIAmfXolz7EnUDn/PQH6kzfdL8TrWJDQae5iv5/19izZV2TW1lN6XQennZZW4/GANUISFQvK9knczEJM9sp+5dp1dkfsorKw3AHJyiq32lvg0wCRQegPYs1ZFyTkiUgUgcmdaSnIo5JSFit/9USip7gXru9PCmhUMs8YIOg1RJfrYVZa5wRCgHcEJfJHwtNeoa9h0dvcltc/Fqc2Kptfv038J50RuPUEAkFlhj0nD1w5oTMoC41Uw/IkLpyhjFxIPN2qnfIJp5DpTKpZlAA74N8Hr9Kfm0f8AKQ5Z4A==')
assets['frozen_evaluator.py'] = ('acdccb0f87c690357e54478285476f81a1913045dc0800d7a4ef5df586bcba8a', 'eNqdPGtz20hy3/UrJkjlAnhBmJQsWaJPV+estbeuWj/K0l7qwrBQIDAUsQYBLAaUSLuc355+zAsk5b3EtWsLwMx0T0+/u0dBEHzMyk4W4uPb9+9vRSEfZNW0a1n3Qm5b2ZX0Y1ML1Xdl3lc7IbOuKmUnynV2L0VWF6LJ802b1flOtE21q8paquTkpN31K5g3WguVd2Xbq0Q+ZNUm62XalnWt0jzbqKxK72Wzln23E29e371OP334cCfevf6Yvnn7SYxG7WZRlbm4/fXdu9ef/pH8pnDFUduVD7CO+HTz8ZfX/0jq9svJr0rCh0Kq8r4eNTXgaSe/ubl9+7f3PLdvBCBRFji7rNtNr2Kh2qrsRVnARsu+lIr3tFCyAyAlbn3Ttk3Xi8cSdrTpRSaWZZ+cGCzzpu6zslaiX0mx7JovshbrrFWIF621zupyKVXPKLRdU2xyoPhiZymz2JRVMSQLrXASBMFJuSboWXffZp2SJwBiDUCrSuaInhJ6wI/Npu5lx99hh1leZUpJ+92+MiuuMrWqyoV5ROx4cpv1+MFM/AiPJ2ZUvVm3O5EpUbcnPNpsItsUZa83Adinqtl0uQe+vAcaDKf0HRBOT1nJqgDqpqrs3aSsb9Zlnj528BJOKnuQqUNTdXmCm0rgBFSf1b2d9uYuFnfpT7/enZycEEdd0ybCNF2WlUzTKOmkaqoHGUYJEBVOXs0m85N3N3c/f3hzC6PDYN0gcYNYBNlDVlbZogQu2eEzMb5lW3xjBcC9jU5uP/7y9g7X+hrQNoOpOL+4vIDhwIHwMJmMx/AAm+3h6cXVy28nr3/5+PNrmDFOJie3Nzdv4MfT8enFCfL52x9f393c0hv4c/L606fX/wD2+9vNLe4uOF1MLl+eLcfnF8VykZ29WLy8WhZXZ1dF9iJfXmTZ5VV+dn5ajIvJZPny8urycnxxOX55WiwmLxbn8mocnNx++PXTjzfekpcvry7Pikt5epaNL8/P5PJi8XKxkBeXV+dX+cviTF6MYe7Z5Kp4cXmxzC8uzorTyYvTF5dX4xfjy0URID0/vf2R6ZkVMl0jsZbwA3Bhn5Igundym+V9ega8A8T7282HdzStBnUDFEqLLY+0zzt+Lko8+VyvnTcqPQUx7DN8UsBb7ilrQfKyfJWuWwUQTk4KuQTRVJ9TVHk5HG64gsWabhdr6ZdFND0R8Aek8MMvt8KME80D6D/NFpUErVci72U9aoCyE323kaIHzQmoAbAYlEC+alDJ8hCB2x9omGXTgV6QRUkSnaDUI9xyKeqmt9jMprEYTeZJVlVhJGBKaL4karMOs22prieR+LM4jZKs3oUae/wDDAgK8u+gf+VN1zVdGNyAds97wKkfk5IC1CqJePWPjWBClHlWgcyt20oixXAd2hUczJs78Qw0QJJ1WX0vDeEStcpaCZLEg9cyq/EQDZYwheZHA3Sfi2O7oAVyiQoNJsIiDHlEiyIh3je1nEewpJlNMzrZb7oaMZOgkWG5oO7jui9Gf6kLYAKzYKx3uEPwMFj9vgG2Cs3nAYIWmGYZI+LpUmYATKpQ1g9l19RoKWPLJI513jPLMpugKc3gVOsvsmtGlazv+5VQ8p4nb4Bl74lFQBY2QH1YC/6ejEc8mQ+jEGxjDZuA1kRLBlSCreC6KgxhZR+vKBb4BsUqipi4JH7+nMMpRb9r5fWiaSqegmxaxgwdzKeQYAxkB7Z0MM9xHUoXQKDxSEb4bzIG/lW4bujWxT/ZYOBoAn+d2qHLqsl6N1bTy58xmR6ZAOyS2Ul1061Peb/6uPUy3mmfOhifpWxhNG5BYzQXf7KPkyk+hbzmX8TYzdMyi9P3hZBYGpyFst5I+/JfxafmcbTOfgPaKqltOsD5DCQmlSjIJwHtsASlg6zBDhgwFHKwZgYSYb2fxNE0dqxltp/NELW5/WIe6Ts/2PnLLmN0iGp5VbbhCMmH5ALJs2s7UaZVYjBgQKLEESWvGoUYA3jxg13WSpZby874DcYCx4ekY+7XZR16YsqrDaQ0csAKWfWw8x41EzGJHj/7zdv1b3Mj+V3PxwivvDW0WeGdA4Wz6j7BYSGt7gb2W4C0g2EanKO9NjfwiZgxNIphVs7FXxlJQMEAipBxLFTgKCEr0Nhjd5haynH6tZg907s0UwCL7bN+Cxzf7571YMJOn+EL+MEg4o6VJJ/Xues0K2rFqYHEPEZrvKzrsl3K/ltID0pzNe5Oe5Go+0/PL0KnKT7LHSoJBQ6ZLIbzjMDzS+C6ncNulWxa9M9DeJvIOm8K8NKAaxbBf48D+DdLSC0lEJAc+4zuYVKAk6rCjO1RdDAsOgSGfKZIOO83DTjgiFiYRUnfLHbgooWavzSZVslKbjVB0JX4q/Ot6W/wxcHg97xZ2IeagpQqZg7yb6fIVnVBYNjUgXAdvqXQ5PC1Nl4paqODj8sNWqUnXh+fYljz4IPv+h6uqO3fVKDfwg4LB1E7/YrZB7i/SDnYCpFOMUZHwE6dZoZBRPBEJGQce1qLXfP0sayL5lGxq6FDLOApYgAcB1ZQAwI5CwZBWIABAKzUy21vjhaitRQDH1L63jQO5NjSckijReGQ9z3eM+BmgQ6DAlQ+sIUU2eEa5Q6VEQQfEOJ06joMYnRSp0Hk2DUacJk2Lt7KINd9A4FgMBf/Av66Jhf4CDJFdkop7ACHIiXbIdFLfpgE6DgO9wGTB87/wGDBaAcRacR7ZZgaOUO66Lse56+1ND4no6rRcIdHsFoIP3d4fiJfyfwz6Hgtrxh+kzaGbxYk6Leqah6BY/LPlbz+KQOtGWF4irFi4Wkb0jQYjQH707fZ5zmrKVJS+CbB4FB9M5Q+pvZw037o9b393q0kbaeXNRy8AmIt0YQTu6MdR4dfgrpAG86USASmMbw0AksN0gVfgkd+j5YIjAcySCc8rysw+ilvugL3eURSSPoiq5PIfYKtfQGzrhVxgHwTzEHr0RdwAc2HVnbA3GlZHP3aj73XWp5Io+nsinK6XoNwc2lAMDdURy8UsSNS44OSPb+gkMc+4Wd80DumjzoDEjJMhxBxOIXj3zuvd3BWRBfK/yyXZV4CrV/Gp5dnI6YgeG8rTmXgSQbO1OGeYjitZYPMdET2E1D6a+U7g2BrilI7Vzg96QjrMEhBD0DUMZ77LmWo2RBPENUSKGXwDwL4McTJYNiCpN8CF9BeEZEZj7HiOhBqLdh7iz6A+m5oUYfcc9QrqOJYbbrV6W1H4auFsefrHpB4GXwwZCXknnuLWNuhqauIsF9xd988g80BEZMMuIp/cIdqFJ0waDrVNxmvvnhksLroesDFMwIwBw9snW1DO5vcM7c6+WY2cEI/laaRQxodCZzYviSY6wF9nGBaDiwBKxgJTm1lBVBrZlmBqC8qaRGKLcKR9SJiy6UpczyHtfw3WfpYm/3YN/+wYxT5Z0Z0BoqQJe3kjxE2EPdQcNjpeJQ9KXwR+TkNzGEckZNZPocnGAz2BQ6XmSAHLYK4IdZGqyCA75sbFGccBsYAk8KoJA91sBVzlmyT5gCzslrrWBEcu8+U4IiGwWU8/EqkjQ5jWyA8UH+wGJ/F08t5Z3VkQSQgh5bL9V7eZ3JOahDpC0NKtQTzD77tarZazyOXNtr/DvBmSzviezT92eWFYB3GE9QlLDjMlFsRNnTOunxVYgqIGZscStFtgMEjG0gASo8rELBwtdbJAgoOYzyNcWKp6Y9c7o0kYpuxDzp89/J7eKw6N9V4gW1TbdagxYj4s5UJ9zFHjPFDaE0Spt8iXMMMgThJB42YFYrRpCuOOb9+i+F/ayBqcMqQhUPOHqOrt9pUVeBTW6pN1cc2L3OYZ7K2OAXDn2Kwg8siEgfpByN4WlhpTWdy15SftwsMeOyPdfit8draQXwwPHTw7zA1SrkJHXOTDH9FiL5Cx23O9D40+fgRSDAgiR+QVPLYyT2pQ93Kmvz+GzqH+fy7kmbButzP/wzeTSfz7wrOL2S2bKnLVZK0J2xNi6bMIlNHt8hsGzs6aPYzPCKuKfIin05XM2IIGfQsXDU+XJRf00EY+syjPxhnqGa8PWu9sd6B3MWHYVyD6cBrxtw8n5ExAW7cIBKJD7wWCshSLw6ZPhGeUP4fHOWsSymmlN2xGRx2ukmH8HTRsvGma7dpWFGKsDiBtbGAAuUUtE6n0DlKjywKZhfzzBgLumWfDiZR+fzfosVvfraCkxFmFbSc8dA7R5UWG0sF4ubxV2xZK7ZnbIony7JPYeXyS1OrcBujuJYFqBstsLGgANSroYAuMDEp12jhqVagH9foPIPnnsu2J/e7K4t7iJCwzMLWBUwHjIPjxApFuZY2+V2n+AjcmnoyYDCxBYlYbO3Pfm0CCwXrFkxDeLiO5mwFkiu9oWlVfpYhzteiWmbq6EqneoFHWd6vevUH0Nx4tBc96kqurfBQT7vozDRREhQRZ/NAB/XzfzYJfUSpv2/s0fiVKXbAKgnR+Nf+h4mvuLdw4ijuW5NBNkSn5wE6VLPpUaNvtwk+sM/iZc1VX/BX+OHwI54Az7eWHyf8RUzkaDKOcfYw34xnwhN2x+F9wZLSdmvqSf0ck9cGjh2lT86C1nlgrhx/Se7EX2GhHyhK3W6jZ1S+fYaHvMNjs3wUCx4b7gCeRm2YTwSVCWigZoJ/UHMhJvBI/2LJ2QgHvMMF4JXGDV7on74ZqdRighKZgyivQTtUmgF0jZGinvwB1Hu7C48yHXy0tbzpHtVGtOKMUZ7jXp7rN4z23CehhQgcCmT84VrosW5LOB6o+AUopL+ZvdmVNJnsYnqrEplXhS1V9Ppug+kgEgeNs0mOq8M0Ps7BZDlP8os/VJy95nUsEWDoBP7X1QiIDK3UTdkP9MuWWSF9VtXK0CJDHupeCVRD26t/umrmnqNTSG+9mavDIici/miN8N95vIeGLiH7s80bzO6CTpvb0rjEjp8UT1nh5I002XvDtOimTU11Y+ZHFA/RnOUuirQX5X1ijcQxNPrt1kGOH0yUpxsHYgab3EWGuXWAEeYxufamDEFRL9kvMlzI3YmfJJ/hX7DHPPFiK/fSd2P5reWE1LjjMNHnBscJGD0d5wMr3Dr/BrKKqQIEYDMFlA7DLybVlVOdTTEaPKLBom8v0/NU70kvhE8Us00is95AzbuZZ+OUN65n4maGEw/6Mp7GeQhj0LoxnKQpg6HUAIod/t29z9z0+QFUD1E0TqlKSQBgrVnXbOoifHP3jNIydlyCNUREASO4J8fAefKYuclFuq8ey+7TgJlK5ViPBSGV4KIudoQZYEScYyR7HLn06XCR/aiFaUisCWQYezRkH3343acx+ebDz6fz46e3aPrV/krT+ZA3nOyt1xl4e9JIHxhe3LFTCxitgajsC6kJpni0bokDftrvUaH5bFcU5n8oVz9UQ1osOF/F+oJXNQnWbwfLtdTfmHJ0iiGWXntAiGXwdf0tBRbZqPTr4luwB5dhzNZzhj/Szwv9HA3XwjJGvKBwPzzoFhu2k0VxeKx77MioYReaaU+LDoML8+cIaDvpCaDed9P+FjSVghDnAQLTYVDB5NWsQeUmsuggKVwi3ws1XBRwSz0s4nHVwN+YYNS5XIyJsacEa+bLcotFEZj/igP17oHrIvjq35Xt+RHrcks5KPYYbFDQgRdLFhi0QtGsE8AxA2xTeB9iYx0fWNFlj4OuGddn55wiSiG6MjaqqYPko0lK67Qn/2C/buryd6z2lVuGxc+cybTZXaYq0BDDRkn1Qc8BBqHTiC5KCPc2IK7lNnolegiNqsMvsfEJr/koNBSvH2PVKInxDxAkwXO7B6jhOEaFx+hFMQK99kjif/S7LYiI4Nf5KfDnpNGZTNj1RWjOGOrcd3Oe09aOfRlaUMxDr7F1eB0Yj4O3ZtwMUHst+pMQoEI4fnVOA4Eqv28yehUSnvEsGZ+ex8nVy/P504pYSlDbyCZx0EkIkHMADGbKkQI14sdPH+4+/PjhF61KAk4gA9DggynTAec+lM1GQaDbyY062mRNAfhUnMcXlxc6pHsuJvFkPEafEH5+cfWSpCQWL16aqH0JpoKZLRHBEPmfQFTFZCx+/uKERFUNhJ/hOHkhFAAEJ8+0JNox8bD/j50evwuQlkDNPTl/fjY2qV/ufDxA4nVVof5cNQXAW2Uon+wYvAKBVdTjpj0ZUSqBssfFXbIKGJdigzamZxEgqQhcQ5sPTBwkAZ6ZNplIc1AqckSVsGGCwZVswN3tTJNmxjoHFIntpbLBL2j9+3vKOxQix0ZPmrNusN8KJFhJrI0dbvkGRu5MQRsDdjz9XBaS28IB+L3sDdVpW34nF/Y+axoBUpOr0bLDhLFpqxSvzeEoYJ16Hzb68iD8/iZs8ylRrpO/b9AIJuIdBFBZLZqlXuk/bv4uCBZ2mDVYAFTiXtYSTl38mNX1Trf5Y2cM7O1w31jkho8PcivQ7RAbpY9IgGcxSN+0SAygblZTIhj3b0/nlUckIADbhrdvkDvkNq82BeK+B/l942WD3TljumKYbIVFMBkicXc0AgJHzMI8b1qgQLvaKWbxNsMK/HtZEuJUese5mYCjLZclti1sugUCeMyqzwR2gX5k1hl+pPBVkQ7w24UpIe5w9cTKdQ7rfk+/a1hu4TzbpuLkF1VSxolAgpP8A7dI2ADm5A4ZAnivtwkZnU1bYh0ezmcqwLfjNTxUtjsshoF5M/oFhZFFHjXCYEMYV4AXDKi/2gcdnkbYpSpNgaWtNigBnQT4VXavpj5Mr6AKSh8h09EB8+jyATOfqR8cfCaOM18PUTnTqBBg2gmtx5pBN2KCerac8kqELw6mIIwnZxww5e0GFCpYSDgTJBK4ywqTaKTdBsWSfgXLuDJ7wwKhuEyEGVGtABdUJvBnguyg1y4WHWqTIybglrpIRX3QdgzSbDpnF7J/lMD9eHNC5pserQYrfdNibK7yIPf3JXjNr8BugYMHPjKqm5xKO8/NepTQBpRkBjyDDauHCpLS1HDkwM8IbbvzmxfB6FNNsYTFeOdh3ijB3fvoSdb25w3sGmO9gkvy5C8VZNEKIe8bx6V985h1Bat7JsI+g4xR8nL4lFNqW4tjBGpSWymrQ1CXA9aKbAA5i4az5fB0iHG0OjDqGBXCDbq1oHnAdmJYqFCzgECiaOu8vkSdWTO6OAT04D3FqSNKUo2cArFqvgOTyrxFZT7j9e3THlD5JJEdNz0qa8wkUaLxedu0m4rJfftG8yBEuLRsePvmz9eUX2WdDuGg+Mj5N6dYtkhl1A14LKCMgIO+wI4OwNtkX6xde0q/cztvAVKtwIPEWy/6fkHeSKrLk4OkB2HWDrRzI/oNJ6k7kcMxqBFf38LPCD3jCxTsHqDDgMT/T+7eef3mBg5LdqADlLjZgAtQIBqURdy/0zFwcGIkTO3mam8mEdbV+wmWJjLZ6x0DGgQsW2zg0Nm/B86sJGy7WdJhvxJZwaqA9Bu4nVhG5/7vs2QMfhtCIDWEQYk7eIR3NsYfaO29ow++60v5rPcK9dW6qUdY1jVlfLJ+7PGAkQP553IZiq/aLJA7W03qtgPVSomp4I6VToNKhAk1Yr6EZzwBF4iLkPW7lTKKv4fyhFvGZGbziDZZoeICVjrgcDhiFz0CTXtjJZnd0G5wOYPVm0BTyF6Gru8hhVwcigdQgTvA7e0QhWiBg8PRd3ysQ7uPCFVnCA0TqxoeQeKbRjKmCKBg2cpc/CCsFw2422j9OfykJEQiTsfxGOICjmQEhil0PSw+wMFFQuLq/N+ECcydvUFdWxv9w15LIj5JdjW4LQjwYQNIDnzhlM46QxuEVxiFcUDQb9rH4We0zZ4N5IT0iAThOTM2sHSMNb4Rt+mQw008aPjPJIyQYQ0XkgekNJ10WMVx9z3y5eACq/b4kEdNEsvcQcXkFN8WTYnFTAqL4GIN2zbXTk2kh7fN3Ixg6j1gUk4XR4Npnpif95MzgS4JpQVeq8XqMDiLXz9PXW1SJ7a2aOXzxFSQbHLrYEGu38IiauqybnnCr6+vFWfKyOHmBshv33RRs+FKqe5RMTdqcefUoDnVudjY6/yc2nau2WeLK2OKteV5FGML3WDU6bFR+ykr3QI6NXjH3AGGz5RBCpzLSIl7/OJn9+PAy+rjR+/xENggkTYdJv9hA2hrU+OluMXiwPgV+M78/M1vafP4wWvzfKBihL38MY3NjRdMP7vKzSSm+6o/TKJoRhlmPXC/kEZKDE/J5uWm+YNLenIGASEaFtCXT08G2UmKzZ/ismGtGA3EGkHuVfsNYUb5QzykOFf9h4s45GcMHFOwXqUSFmFA+/OILbFaiI3QafCDm709NtJc5/jK434IYAqVqR72qkwMzQpWNKAPDaMKYVn7uB+nkGu5skWuli4R7rXTfbf6XvNMH9r0sFnKia7tkfJroJZZBwfyBEkdJE0jXA1fOmbiSw4+L+FlZY8lyehjUVgjtZ9yxkqZ+TSQu3k0o6SW1+XMundmXIkUv6c6eZ8V1BC6l2fWI2OtJ2jFeWyKSPQUnRyyklbOdJE8mPtdql7LjafEh7c2vCYN1XJLH97njpk2/uXHAelU63V+o5HD5kTxp/3ioyu/+GlQjzyqpeJFkGGJx6/IrGNdjAFbtNilrEHJLHhj/hRqWqFpMBOcifBy3HpcdGhzBrUjupwzRIS3Z7H5w9npU9jyQv8flPcKFUi42Jq4Q23t6nrgJMDnNEVxSFNg+BRQLOs0DXT/BP4Chu7a/CqG5HV3v0FX4yO9D62D3NTXaVo0eZpG3rwEfPw001PCANvu0w48vSCmdnHsJ/vOcH0p6Z8bbH75hTc6NllAr7jw1FwmlT95Jav2GvXUiG/aiPcf/4tdZR10xtSsQk5yXZBXp13ds/EIK6K+Zgu+C937JR5BzDdFrwO665rizXo9GWaoa70C/YNrqHDYZA5vEv+g6aI7vrTMoOjcndTqFSW3bX7Eu3zu940sqekZ/FoXNJh78RR9qetZ636zBN/qJxXBMOlEYh8+dUe0JkPpDD7e+DBFcb10FP3LNb4yj4LyyLsQGciDiQbL/EoLh4CepGPHpndIaa5irCxHRtH+HbDwn6Kl2YVfHBtSFC842Xw13zalXzei8AEbNunXlOiLT6/2fqVLDSGPzi5bBo88xz3WqFxbL9+/ezjcYuxvP4r3t2f56Mjupt5V46Wg302S6mEhXYMb2nySEnyfYMI5DB4XAd9Q6zuZrQ/9A7wHADHhlxSjJzhZhfqNxsbPnhnGcRfbvN+QEvrnEQ8Q072M5heoDDiSiReZQACl8O8m5b3O+nxlI3k1DWLQ0t3MtrSwf6/ijnrHtAnXUcncxS3LaqNWnuJ5QkCnBv5H7VqQC8JOACYPAPwfugkDWP8LVS6zTA==')
for name, (expected, text) in assets.items():
    raw = zlib.decompress(base64.b64decode(text))
    assert hashlib.sha256(raw).hexdigest() == expected, name
    (folder / name).write_bytes(raw)
env = {**os.environ, 'CUDA_VISIBLE_DEVICES':'', 'OPENBLAS_NUM_THREADS':'1',
       'MKL_NUM_THREADS':'1','OMP_NUM_THREADS':'1','PYTHONDONTWRITEBYTECODE':'1'}
def launch(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=folder, env=env, check=True)
launch('test_comparison.py', folder / 'frozen_evaluator.py')


In [ ]:
root = Path('/kaggle/input')
lock = json.loads((folder / 'input_lock.json').read_bytes())
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
source = None
for name in ('pinns_map_axis_inputs.bin', 'pinns_map_axis_inputs.zip'):
    for candidate in sorted(root.rglob(name)):
        if digest(candidate) == lock['private_input_sha256']:
            source = candidate
            break
    if source:
        break
if source is None:
    # Some Kaggle uploads auto-expand ZIPs. Rebuild only the five verified input members.
    for candidate in sorted(root.rglob('legacy_replay.npz')):
        base = candidate.parent
        names = lock['member_sha256']
        if not all((base / n).is_file() and digest(base / n) == h for n,h in names.items()):
            continue
        tmp = folder / 'restored_private_input.bin'
        with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_STORED) as z:
            for n in sorted(names):
                z.writestr(zipfile.ZipInfo(n), (base / n).read_bytes())
        if digest(tmp) != lock['private_input_sha256']:
            raise ValueError('Expanded archive fingerprint differs')
        source = tmp
        break
if source is None:
    raise FileNotFoundError('Attach the private pinns_map_axis_inputs.bin from this delivery as a Kaggle Dataset.')
common = ['compare_axes.py', '--input', source, '--lock', folder/'input_lock.json',
          '--original-evaluator', folder/'frozen_evaluator.py', '--protocol', folder/'PROTOCOL.md']
launch(*common, '--output', folder/'checked_inputs.zip')


In [ ]:
destination = Path('/kaggle/working/pinns_map_axis_comparison_results.zip')
launch(*common, '--score', '--output', destination)
with zipfile.ZipFile(destination) as archive:
    assert archive.testzip() is None
    assert 'private_replay.npz' in archive.namelist()
print('Verified private result', destination.name, destination.stat().st_size, digest(destination))
